In [1]:
import pyodbc
import pandas as pd

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [2]:
euro_govs = pd.read_csv('gov_fut.csv')

In [3]:
euro_govs_isin = tuple(euro_govs['isin'].unique())

In [ ]:
# parameters
eurex_lei = ('529900LN3S50JPU47S06', '529900VIIHQECIS4ET47', '529900UT4DG0LG5R9O07')

prerefit_start, prerefit_end = '2018-01-01', '2024-04-26'
refit_start,    refit_end    = '2024-04-29', '2026-09-01'

schema = 'xlab_ecb_prj_sftds_cb_common'

In [ ]:
# gov_fut.csv as an inline SQL table, so the CREATE TABLE statements can join to it
gov_fut_sql = '\n    UNION ALL '.join(
    f"SELECT '{r.isin}' AS isin, '{r.country}' AS country, '{r.expiration_date}' AS expiration_date"
    for r in euro_govs.itertuples()
)

In [ ]:
# pre-REFIT query (crp_emir_ecb)
def prerefit_query(start, end):
    return f"""
WITH gov_fut AS (
    {gov_fut_sql}
),
base AS (
    SELECT
        -- identifiers
        e.tec_ruti,
        e.reference_period,
        -- counterparties
        e.leg1_counterparty_side AS direction,
        e.leg1_reporting_cpty_id,
        e.leg1_other_cpty_id,
        e.leg1_reporting_cpty_id_country_code AS rep_country,
        e.leg1_other_cpty_id_country_code     AS oth_country,
        CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP' ELSE s_rep.sector END AS rep_sector,
        CASE WHEN e.leg1_other_cpty_id     IN {eurex_lei} THEN 'CCP' ELSE s_oth.sector END AS oth_sector,
        -- product
        e.leg1_product_id  AS product_id,
        g.country          AS product_country,
        g.expiration_date  AS expiration_date,
        -- trade terms
        e.leg1_notional            AS notional,
        e.leg1_notional_currency1  AS notional_currency,
        e.leg1_price_rate          AS price,
        e.leg1_quantity            AS quantity
    FROM crp_emir_ecb.emir_ecb_states_deduplicated e
    JOIN gov_fut g
        ON e.leg1_product_id = g.isin
    LEFT JOIN {schema}.pth_sector s_rep
        ON e.leg1_reporting_cpty_id = s_rep.lei
    LEFT JOIN {schema}.pth_sector s_oth
        ON e.leg1_other_cpty_id = s_oth.lei
    WHERE e.reference_period BETWEEN '{start}' AND '{end}'
        AND e.leg1_counterparty_side IN ('BYER', 'SLLR')
        AND e.leg1_notional > 0
        AND e.leg1_notional <= 1e11
)
SELECT
    -- identifiers
    tec_ruti,
    reference_period,
    -- counterparties
    CASE WHEN direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS buyer_id,
    CASE WHEN direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS buyer_sector,
    CASE WHEN direction = 'BYER' THEN rep_country            ELSE oth_country            END AS buyer_country,
    CASE WHEN direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS seller_id,
    CASE WHEN direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS seller_sector,
    CASE WHEN direction = 'SLLR' THEN rep_country            ELSE oth_country            END AS seller_country,
    -- product
    product_id,
    product_country,
    expiration_date,
    -- trade terms
    notional,
    notional_currency,
    price,
    quantity
FROM base
"""

In [ ]:
# REFIT query (crp_emir_refit_ecb)
def refit_query(start, end):
    return f"""
WITH gov_fut AS (
    {gov_fut_sql}
),
base AS (
    SELECT
        -- identifiers
        e.tec_ruti,
        e.reference_period,
        -- counterparties
        e.leg1_direction AS direction,
        e.leg1_reporting_cpty_id,
        e.leg1_other_cpty_id,
        e.leg1_reporting_cpty_id_country_code_gleif AS rep_country,
        e.leg1_other_cpty_id_country_code_gleif     AS oth_country,
        CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP' ELSE s_rep.sector END AS rep_sector,
        CASE WHEN e.leg1_other_cpty_id     IN {eurex_lei} THEN 'CCP' ELSE s_oth.sector END AS oth_sector,
        -- product
        e.leg1_product_isin  AS product_id,
        g.country            AS product_country,
        g.expiration_date    AS expiration_date,
        -- trade terms
        e.leg1_notional_leg1           AS notional,
        e.leg1_notional_leg1_currency  AS notional_currency,
        e.leg1_price                   AS price,
        e.leg1_notional_quantity_leg1  AS quantity
    FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated e
    JOIN gov_fut g
        ON e.leg1_product_isin = g.isin
    LEFT JOIN {schema}.pth_sector s_rep
        ON e.leg1_reporting_cpty_id = s_rep.lei
    LEFT JOIN {schema}.pth_sector s_oth
        ON e.leg1_other_cpty_id = s_oth.lei
    WHERE e.reference_period BETWEEN '{start}' AND '{end}'
        AND e.leg1_direction IN ('BYER', 'SLLR')
        AND e.leg1_notional_leg1 > 0
        AND e.leg1_notional_leg1 <= 1e11
)
SELECT
    -- identifiers
    tec_ruti,
    reference_period,
    -- counterparties
    CASE WHEN direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS buyer_id,
    CASE WHEN direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS buyer_sector,
    CASE WHEN direction = 'BYER' THEN rep_country            ELSE oth_country            END AS buyer_country,
    CASE WHEN direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS seller_id,
    CASE WHEN direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS seller_sector,
    CASE WHEN direction = 'SLLR' THEN rep_country            ELSE oth_country            END AS seller_country,
    -- product
    product_id,
    product_country,
    expiration_date,
    -- trade terms
    notional,
    notional_currency,
    price,
    quantity
FROM base
"""

In [ ]:
# test: one day per regime
df_pre = pd.read_sql_query(prerefit_query('2023-07-17', '2023-07-17'), cnxn)
df_ref = pd.read_sql_query(refit_query('2024-07-15', '2024-07-15'), cnxn)
print(df_pre.shape, df_ref.shape)
print(list(df_pre.columns) == list(df_ref.columns))
df_pre.head()

In [ ]:
# explore: US Treasury futures on CBOT, what can identify the tenor?
df_us = pd.read_sql_query("""
SELECT CAST(leg1_maturity_date AS DATE) AS maturity,
       leg1_underlying_id_type, leg1_underlying_id, CAST(leg1_underlying_maturity_date AS DATE) AS underlying_maturity,
       leg1_product_aii,
       ROUND(AVG(leg1_price), 2) AS avg_price,
       ROUND(AVG(leg1_notional_leg1 / leg1_notional_quantity_leg1), 0) AS avg_notional_per_unit,
       COUNT(*) AS n_rows, ROUND(SUM(leg1_notional_leg1_eur) / 1e9, 1) AS notional_eur_bn
FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated
WHERE reference_period = '2024-07-15'
    AND leg1_execution_venue = 'XCBT'
    AND leg1_contract_type = 'FUTR'
    AND leg1_asset_class = 'INTR'
    AND leg1_product_cfi = 'FFDPSX'
    AND leg1_notional_leg1_currency = 'USD'
    AND leg1_maturity_date IS NOT NULL
GROUP BY leg1_maturity_date, leg1_underlying_id_type, leg1_underlying_id, leg1_underlying_maturity_date, leg1_product_aii
ORDER BY n_rows DESC
""", cnxn)
print(df_us.shape)
df_us.head(40)

In [ ]:
# create pre-REFIT table
cursor.execute(f"DROP TABLE IF EXISTS {schema}.hermesf_fut_prerefit")
cursor.execute(f"""
CREATE EXTERNAL TABLE {schema}.hermesf_fut_prerefit
STORED AS PARQUET TBLPROPERTIES ('external.table.purge'='true')
AS
{prerefit_query(prerefit_start, prerefit_end)}
""")

In [ ]:
# create REFIT table
cursor.execute(f"DROP TABLE IF EXISTS {schema}.hermesf_fut_refit")
cursor.execute(f"""
CREATE EXTERNAL TABLE {schema}.hermesf_fut_refit
STORED AS PARQUET TBLPROPERTIES ('external.table.purge'='true')
AS
{refit_query(refit_start, refit_end)}
""")

In [ ]:
# full series: pre-REFIT 2018-01-01 to 2024-04-26, REFIT 2024-04-29 to 2026-09-01
cursor.execute(f"DROP TABLE IF EXISTS {schema}.hermesf_fut")
cursor.execute(f"""
CREATE EXTERNAL TABLE {schema}.hermesf_fut
STORED AS PARQUET TBLPROPERTIES ('external.table.purge'='true')
AS
SELECT * FROM {schema}.hermesf_fut_prerefit
UNION ALL
SELECT * FROM {schema}.hermesf_fut_refit
""")

In [ ]:
# check
pd.read_sql_query(f"""
SELECT MIN(reference_period) AS first_date, MAX(reference_period) AS last_date,
       COUNT(*) AS n_rows, COUNT(DISTINCT product_id) AS n_contracts
FROM {schema}.hermesf_fut
""", cnxn)